<a href="https://colab.research.google.com/github/fdac25/MP2/blob/main/Vis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt



In [ ]:
df = pd.read_csv("your-net-id-here_project_summary.csv", sep=";")
df.columns = ['project','commit','author','time','message']
df['Month'] = pd.to_datetime(df['time'], unit='s').dt.strftime('%Y-%m')
df.head()

In [ ]:
prj='mmorise_world'
df1 = df[df['project']==prj]
df2 = df1.groupby('Month').size().reset_index(name='Commits')
monthly_counts = df2
monthly_counts['time'] = pd.to_datetime(monthly_counts['Month'])
full_date_range = pd.DataFrame({'time': pd.date_range(start=monthly_counts['time'].min(), end=monthly_counts['time'].max(), freq='MS')})
monthly_counts = pd.merge(full_date_range, monthly_counts, on='time', how='left').fillna(0)
monthly_counts['Month'] = monthly_counts['time'].dt.strftime('%Y-%m')


In [ ]:
# Plot
import matplotlib.ticker as ticker

plt.figure(figsize=(10,5))
plt.plot(monthly_counts["Month"], monthly_counts["Commits"], marker="o", linestyle="-")
plt.xlabel("Time (YYYY-MM)")
plt.ylabel("Number of Commits")
plt.title("Commit Activity: "+prj)
plt.grid(True)
plt.tight_layout()
ax =  plt.gca()
for label in ax.xaxis.get_ticklabels()[::2]:
    label.set_visible(False)
plt.xticks(rotation=45, fontsize=7)
plt.savefig("your-net-id-here_timeseries_"+prj+".png", dpi=600)
plt.show()
plt.close()

# Identify the Longest Inactivity Gap

2022-02 to 2023-04

Add all that info to netid_project_stats.csv

In [ ]:
import pandas as pd
import numpy as np

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt

NETID = "jmill291"

github_stats = {
    "neuroelectro_neuroelectro_org": {
        "nstars": 19,
        "nforks": 7,
        "lastGHCommitDate": "2024-12-07T16:21:16Z"
    },
    "ropensci_textreuse": {
        "nstars": 200,
        "nforks": 34,
        "lastGHCommitDate": "2026-07-25T01:12:17Z"
    },
    "freesurfer_freesurfer": {
        "nstars": 877,
        "nforks": 301,
        "lastGHCommitDate": "2026-09-29T16:24:42Z"
    },
    "openforcefield_openforcefield": {
        "nstars": 415,
        "nforks": 107,
        "lastGHCommitDate": "2026-09-28T19:46:56Z"
    },
    "markmfredrickson_ritools": {
        "nstars": 17,
        "nforks": 11,
        "lastGHCommitDate": "2025-05-17T13:18:53Z"
    },
    "annkennedy_tracer": {
        "nstars": 57,
        "nforks": 9,
        "lastGHCommitDate": "2025-01-22T06:50:08Z"
    },
    "brentp_vcfanno": {
        "nstars": 403,
        "nforks": 56,
        "lastGHCommitDate": "2026-06-16T19:02:01Z"
    },
    "r-spatial_mapedit": {
        "nstars": 221,
        "nforks": 33,
        "lastGHCommitDate": "2026-02-15T11:31:59Z"
    },
    "isa-tools_isa-api": {
        "nstars": 48,
        "nforks": 42,
        "lastGHCommitDate": "2026-07-23T08:16:52Z"
    },
    "nvidiagameworks_falcor": {
        "nstars": 3242,
        "nforks": 622,
        "lastGHCommitDate": "2026-09-21T15:08:34Z"
    }
}


def find_zero_gaps(monthly):
    gaps = []
    start = None

    for i, count in enumerate(monthly["#Commits"]):

        if count == 0 and start is None:
            start = i

        elif count != 0 and start is not None:
            gaps.append((start, i - 1, i - start))
            start = None

    if start is not None:
        gaps.append((start, len(monthly) - 1, len(monthly) - start))

    return gaps


def classify_activity(monthly):
    y = monthly["#Commits"].to_numpy(dtype=float)

    if len(y) < 6:
        return "irregular"

    third = max(1, len(y) // 3)

    first = y[:third].mean()
    middle = y[third:2 * third].mean()
    last = y[-third:].mean()

    overall_mean = y.mean()

    x = np.arange(len(y))
    slope = np.polyfit(x, y, 1)[0]

    if overall_mean == 0:
        return "steady"

    normalized_slope = slope / overall_mean

    if (
        middle < first * 0.65
        and middle < last * 0.65
        and last > middle
    ):
        return "U-shaped"

    if normalized_slope > 0.015:
        return "rising"

    if normalized_slope < -0.015:
        return "declining"

    rolling = pd.Series(y).rolling(6, min_periods=1).mean()
    changes = np.diff(rolling)

    sign_changes = np.sum(
        np.sign(changes[1:]) != np.sign(changes[:-1])
    )

    if sign_changes >= 6:
        return "cyclical"

    if np.std(y) / (overall_mean + 1e-9) < 0.55:
        return "steady"

    return "irregular"


df = pd.read_csv(
    f"{NETID}_project_summary.csv",
    sep=";"
)

df["time"] = pd.to_numeric(df["time"], errors="coerce")
df = df.dropna(subset=["time"])

df["datetime"] = pd.to_datetime(
    df["time"],
    unit="s",
    utc=True
)

df["Month"] = (
    df["datetime"]
    .dt.tz_localize(None)
    .dt.to_period("M")
)

project_stats = []
interpretations = []

for project in sorted(df["project_wocid"].unique()):

    print("\nProcessing:", project)

    project_df = df[
        df["project_wocid"] == project
    ].copy()

    monthly_counts = (
        project_df.groupby("Month")
        .size()
    )

    first_month = monthly_counts.index.min()
    last_month = monthly_counts.index.max()

    complete_months = pd.period_range(
        first_month,
        last_month,
        freq="M"
    )

    monthly_counts = monthly_counts.reindex(
        complete_months,
        fill_value=0
    )

    monthly = pd.DataFrame({
        "Month": complete_months.astype(str),
        "#Commits": monthly_counts.values
    })

    timeseries_file = (
        f"{NETID}_commits_timeseries_{project}.csv"
    )

    monthly.to_csv(
        timeseries_file,
        sep=";",
        index=False
    )

    plt.figure(figsize=(12, 6))

    plt.plot(
        monthly["Month"],
        monthly["#Commits"],
        linestyle="-"
    )

    plt.xlabel("Time (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title(project)
    plt.grid(True)

    step = max(1, len(monthly) // 15)

    tick_positions = range(
        0,
        len(monthly),
        step
    )

    plt.xticks(
        tick_positions,
        monthly["Month"].iloc[
            list(tick_positions)
        ],
        rotation=45,
        ha="right"
    )

    plt.tight_layout()

    plot_file = (
        f"{NETID}_timeseries_{project}.png"
    )

    plt.savefig(
        plot_file,
        dpi=300
    )

    plt.close()

    gaps = find_zero_gaps(monthly)

    gaps_at_least_3 = [
        gap for gap in gaps
        if gap[2] >= 3
    ]

    if gaps:

        longest_gap = max(
            gaps,
            key=lambda g: g[2]
        )

        start_idx, end_idx, gap_length = longest_gap

        gap_start = monthly.iloc[start_idx]["Month"]
        gap_end = monthly.iloc[end_idx]["Month"]

        commits_after_gap = int(
            monthly.iloc[end_idx + 1:]["#Commits"].sum()
        )

    else:

        gap_start = "N/A"
        gap_end = "N/A"
        gap_length = 0
        commits_after_gap = 0

    number_of_gaps = len(gaps_at_least_3)

    activity_pattern = classify_activity(monthly)

    ncommits = project_df["commit_sha1"].nunique()
    nauthors = project_df["author"].nunique()
    max_time = int(project_df["time"].max())
    min_time = int(project_df["time"].min())

    gh = github_stats[project]

    project_stats.append({
        "Project": project,
        "ncommits": ncommits,
        "nauthors": nauthors,
        "from": min_time,
        "to": max_time,
        "nstars": gh["nstars"],
        "nforks": gh["nforks"],
        "lastGHCommitDate": gh["lastGHCommitDate"],
        "LongestGapStart": gap_start,
        "LongestGapEnd": gap_end,
        "LongestGapLength": gap_length,
        "NumCommitsAfterLastGap": commits_after_gap,
        "ActivityPattern": activity_pattern,
        "NumberOfGapsInTimeline": number_of_gaps
    })

    interpretation = (
        f"{project}: The overall activity pattern was classified as "
        f"{activity_pattern}. "
    )

    if gap_length > 0:
        interpretation += (
            f"The longest continuous inactivity period ran from "
            f"{gap_start} through {gap_end} and lasted "
            f"{gap_length} month(s). There were "
            f"{number_of_gaps} inactivity gaps lasting at least "
            f"three months."
        )
    else:
        interpretation += (
            "No zero-commit inactivity gap was found in the project timeline."
        )

    interpretations.append(interpretation)

    print("  CSV:", timeseries_file)
    print("  Plot:", plot_file)
    print(
        "  Longest gap:",
        gap_start,
        "to",
        gap_end,
        "(" + str(gap_length) + " months)"
    )
    print("  Gaps >= 3 months:", number_of_gaps)
    print("  Pattern:", activity_pattern)


stats_df = pd.DataFrame(project_stats)

stats_df.to_csv(
    f"{NETID}_project_stats.csv",
    index=False
)

with open(
    f"{NETID}_part2_interpretation.txt",
    "w"
) as f:

    f.write("# Interpretation\n\n")

    for text in interpretations:
        f.write(text + "\n\n")


print("\nCreated:", f"{NETID}_project_stats.csv")
print("Created:", f"{NETID}_part2_interpretation.txt")
print("\nPART 2 PROCESSING COMPLETE")


# Interpretation

annkennedy_tracer: The overall activity pattern was classified as declining. The longest continuous inactivity period ran from 2019-08 through 2019-10 and lasted 3 month(s). There were 1 inactivity gaps lasting at least three months.

brentp_vcfanno: The overall activity pattern was classified as declining. The longest continuous inactivity period ran from 2023-12 through 2025-06 and lasted 19 month(s). There were 8 inactivity gaps lasting at least three months.

freesurfer_freesurfer: The overall activity pattern was classified as cyclical. No zero-commit inactivity gap was found in the project timeline.

isa-tools_isa-api: The overall activity pattern was classified as declining. The longest continuous inactivity period ran from 2025-06 through 2025-09 and lasted 4 month(s). There were 1 inactivity gaps lasting at least three months.

markmfredrickson_ritools: The overall activity pattern was classified as cyclical. The longest continuous inactivity period ran from 2014-10 through 2015-06 and lasted 9 month(s). There were 9 inactivity gaps lasting at least three months.

neuroelectro_neuroelectro_org: The overall activity pattern was classified as declining. The longest continuous inactivity period ran from 2020-05 through 2024-11 and lasted 55 month(s). There were 4 inactivity gaps lasting at least three months.

nvidiagameworks_falcor: The overall activity pattern was classified as U-shaped. The longest continuous inactivity period ran from 2019-11 through 2020-02 and lasted 4 month(s). There were 3 inactivity gaps lasting at least three months.

openforcefield_openforcefield: The overall activity pattern was classified as cyclical. The longest continuous inactivity period ran from 2018-08 through 2018-08 and lasted 1 month(s). There were 0 inactivity gaps lasting at least three months.

r-spatial_mapedit: The overall activity pattern was classified as declining. The longest continuous inactivity period ran from 2020-05 through 2021-04 and lasted 12 month(s). There were 8 inactivity gaps lasting at least three months.

ropensci_textreuse: The overall activity pattern was classified as declining. The longest continuous inactivity period ran from 2020-06 through 2024-04 and lasted 47 month(s). There were 7 inactivity gaps lasting at least three months.

